In [1]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\Aswini A\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
pip install tensorflow keras matplotlib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\Aswini A\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import numpy as np
import matplotlib.pyplot as plt

In [4]:
train_path = r"D:\Datasets"  

In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training generator (with augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation generator (NO augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(128,128),
    batch_size=16,   # increase
    class_mode='categorical',
    subset='training',
    seed=42
)

val_data = val_datagen.flow_from_directory(
    train_path,
    target_size=(128,128),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    seed=42
)

Found 4825 images belonging to 5 classes.
Found 1205 images belonging to 5 classes.


In [14]:
print(train_data.class_indices)

{'Ambulance': 0, 'Bike': 1, 'Bus': 2, 'Car': 3, 'Truck': 4}


In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),   

    Dense(5, activation='softmax')
])

C:\Users\Aswini A\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8   
)

Epoch 1/8
210/604 ━━━━━━━━━━━━━━━━━━━━ 2:09 329ms/step - accuracy: 0.4103 - loss: 1.6023

C:\Users\Aswini A\AppData\Local\Programs\Python\Python310\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


604/604 ━━━━━━━━━━━━━━━━━━━━ 250s 410ms/step - accuracy: 0.5950 - loss: 1.0428 - val_accuracy: 0.6938 - val_loss: 0.7928
Epoch 2/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 134s 222ms/step - accuracy: 0.7596 - loss: 0.6358 - val_accuracy: 0.7344 - val_loss: 0.6576
Epoch 3/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 158s 248ms/step - accuracy: 0.8802 - loss: 0.3288 - val_accuracy: 0.7494 - val_loss: 0.6955
Epoch 4/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 193s 233ms/step - accuracy: 0.9579 - loss: 0.1301 - val_accuracy: 0.7402 - val_loss: 0.8975
Epoch 5/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 138s 228ms/step - accuracy: 0.9840 - loss: 0.0529 - val_accuracy: 0.7436 - val_loss: 1.2157
Epoch 6/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 139s 231ms/step - accuracy: 0.9915 - loss: 0.0372 - val_accuracy: 0.7178 - val_loss: 1.4515
Epoch 7/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 141s 229ms/step - accuracy: 0.9861 - loss: 0.0478 - val_accuracy: 0.7245 - val_loss: 1.4850
Epoch 8/8
604/604 ━━━━━━━━━━━━━━━━━━━━ 139s 231ms/step - accuracy: 0.9909 - loss: 0.0305 - va

In [9]:
loss, acc = model.evaluate(val_data)
print("Accuracy:", acc)

151/151 ━━━━━━━━━━━━━━━━━━━━ 12s 81ms/step - accuracy: 0.7087 - loss: 1.5401
Accuracy: 0.708713710308075


In [15]:
from tensorflow.keras.preprocessing import image

class_names = ['Ambulance', 'Bike', 'Bus', 'Car', 'Truck']
def predict_image(img_path):
    img = image.load_img(img_path, target_size=(128,128))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)
    confidence = np.max(predictions)
    class_index = np.argmax(predictions)
    predicted_class = class_names[class_index]

    # Decision logic
    if confidence >= 0.85:
        decision = "High Confidence ✅"
    elif confidence >= 0.65:
        decision = "Needs Review ❓"
    else:
        decision = "Uncertain ⚠"

    print("Prediction:", predicted_class)
    print("Confidence:", confidence)
    print("Decision:", decision)

In [17]:
predict_image(r"D:\Datasets\Car\39.jpg")  

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Prediction: Car
Confidence: 0.99966586
Decision: High Confidence ✅


In [18]:
model.save("model.h5")